<a href="https://colab.research.google.com/github/Joanie-lab/dna-translation-orf-finder/blob/main/dna_translation_orf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DNA Translation and ORF Finder

## What it does
This program reads DNA sequences from a FASTA file and, for each sequence:
- Finds open reading frames (ORFs): stretches that start with a start codon (ATG) and end with a stop codon (TAA, TAG or TGA) in the same reading frame
- Ignores ORFs shorter than a minimum length (default 60 letters)
- Reports the longest ORF
- Translates the longest ORF into a protein sequence

## How to use it
1. Run all cells in order (Runtime → Run all).
2. Upload your own FASTA file using the folder icon on the left, or create one as shown in the test cells.
3. Run `analyse_orfs("your_file_name.fasta")`. You can change the minimum length, for example `analyse_orfs("your_file_name.fasta", min_length=100)`.

## The biology
Cells turn DNA into protein in stages. DNA is transcribed into RNA, and ribosomes read the RNA in groups of three letters called codons. Each codon codes for one amino acid, and a chain of amino acids forms a protein.

- **ATG** is the start codon. It also codes for the amino acid methionine (M).
- **TAA, TAG and TGA** are stop codons, which end the protein.
- An **open reading frame (ORF)** is a stretch of DNA that runs from a start codon to an in-frame stop codon. It is a candidate for a protein-coding gene.

Short ORFs often appear by chance, so the program ignores any shorter than a minimum length.

In [1]:
real_text = """
>NM_000207.3 Homo sapiens insulin (INS), transcript variant 1, mRNA
AGCCCTCCAGGACAGGCTGCATCAGAAGAGGCCATCAAGCAGATCACTGTCCTTCTGCCATGGCCCTGTG
GATGCGCCTCCTGCCCCTGCTGGCGCTGCTGGCCCTCTGGGGACCTGACCCAGCCGCAGCCTTTGTGAAC
CAACACCTGTGCGGCTCACACCTGGTGGAAGCTCTCTACCTAGTGTGCGGGGAACGAGGCTTCTTCTACA
CACCCAAGACCCGCCGGGAGGCAGAGGACCTGCAGGTGGGGCAGGTGGAGCTGGGCGGGGGCCCTGGTGC
AGGCAGCCTGCAGCCCTTGGCCCTGGAGGGGTCCCTGCAGAAGCGTGGCATTGTGGAACAATGCTGTACC
AGCATCTGCTCCCTCTACCAGCTGGAGAACTACTGCAACTAGACGCAGCCCGCAGGCAGCCCCACACCCG
CCGCCTCCTGCACCGAGAGAGATGGAATAAAGCCCTTGAACCAGC
"""

with open("real_sequence.fasta", "w") as f:
    f.write(real_text)

In [2]:
def read_fasta(filename):
    sequences = {}
    name = None

    with open(filename) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                name = line[1:].split()[0]
                sequences[name] = ""
            elif name is not None:
                sequences[name] += line

    return sequences

In [3]:
data = read_fasta("real_sequence.fasta")
print(data)

{'NM_000207.3': 'AGCCCTCCAGGACAGGCTGCATCAGAAGAGGCCATCAAGCAGATCACTGTCCTTCTGCCATGGCCCTGTGGATGCGCCTCCTGCCCCTGCTGGCGCTGCTGGCCCTCTGGGGACCTGACCCAGCCGCAGCCTTTGTGAACCAACACCTGTGCGGCTCACACCTGGTGGAAGCTCTCTACCTAGTGTGCGGGGAACGAGGCTTCTTCTACACACCCAAGACCCGCCGGGAGGCAGAGGACCTGCAGGTGGGGCAGGTGGAGCTGGGCGGGGGCCCTGGTGCAGGCAGCCTGCAGCCCTTGGCCCTGGAGGGGTCCCTGCAGAAGCGTGGCATTGTGGAACAATGCTGTACCAGCATCTGCTCCCTCTACCAGCTGGAGAACTACTGCAACTAGACGCAGCCCGCAGGCAGCCCCACACCCGCCGCCTCCTGCACCGAGAGAGATGGAATAAAGCCCTTGAACCAGC'}


## Step 3: The codon table

Maps each of the 64 DNA codons to an amino acid. A star (*) marks a stop codon.

In [4]:
codon_table = {
    "TTT": "F", "TTC": "F", "TTA": "L", "TTG": "L",
    "CTT": "L", "CTC": "L", "CTA": "L", "CTG": "L",
    "ATT": "I", "ATC": "I", "ATA": "I", "ATG": "M",
    "GTT": "V", "GTC": "V", "GTA": "V", "GTG": "V",
    "TCT": "S", "TCC": "S", "TCA": "S", "TCG": "S",
    "CCT": "P", "CCC": "P", "CCA": "P", "CCG": "P",
    "ACT": "T", "ACC": "T", "ACA": "T", "ACG": "T",
    "GCT": "A", "GCC": "A", "GCA": "A", "GCG": "A",
    "TAT": "Y", "TAC": "Y", "TAA": "*", "TAG": "*",
    "CAT": "H", "CAC": "H", "CAA": "Q", "CAG": "Q",
    "AAT": "N", "AAC": "N", "AAA": "K", "AAG": "K",
    "GAT": "D", "GAC": "D", "GAA": "E", "GAG": "E",
    "TGT": "C", "TGC": "C", "TGA": "*", "TGG": "W",
    "CGT": "R", "CGC": "R", "CGA": "R", "CGG": "R",
    "AGT": "S", "AGC": "S", "AGA": "R", "AGG": "R",
    "GGT": "G", "GGC": "G", "GGA": "G", "GGG": "G",
}

In [5]:
print(len(codon_table))
print(codon_table["ATG"])
print(codon_table["TAA"])

64
M
*


## Step 4: Translating a reading frame
Translates DNA three letters at a time using the codon table. This simple version does not stop at a stop codon (*), so it keeps translating into the untranslated region. Steps 6 and 7 fix this by finding complete ORFs.

In [6]:
def translate(dna):
    dna = dna.upper()
    protein = ""
    for i in range(0, len(dna) - 2, 3):
        codon = dna[i:i + 3]
        protein += codon_table.get(codon, "X")
    return protein

In [7]:
print(translate("ATGGCCTGA"))

MA*


In [8]:
seq = data["NM_000207.3"]
start = seq.find("ATG")
print("First ATG at position:", start)
print(translate(seq[start:]))

First ATG at position: 59
MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN*TQPAGSPTPAASCTERDGIKPLNQ


## Step 5: Finding start codons
Scans the sequence one position at a time and records every position where the start codon ATG begins.

In [9]:
def find_start_codons(dna):
    dna = dna.upper()
    positions = []
    for i in range(len(dna) - 2):
        if dna[i:i + 3] == "ATG":
            positions.append(i)
    return positions

In [10]:
print(find_start_codons("CCATGAAATGTT"))

[2, 7]


In [11]:
seq = data["NM_000207.3"]
positions = find_start_codons(seq)
print(positions)
print("Number of ATGs:", len(positions))

[59, 71, 340, 441]
Number of ATGs: 4


## Step 6: Finding open reading frames
For each start codon (ATG), reads forward in steps of three letters until a stop codon (TAA, TAG or TGA) is found. Each complete ORF is stored with its start, end and length.

In [12]:
def find_orfs(dna):
    dna = dna.upper()
    stop_codons = ["TAA", "TAG", "TGA"]
    orfs = []

    for start in find_start_codons(dna):
        for i in range(start, len(dna) - 2, 3):
            codon = dna[i:i + 3]
            if codon in stop_codons:
                end = i + 3
                orfs.append((start, end, end - start))
                break

    return orfs

In [13]:
print(find_orfs("ATGGCCTGA"))

[(0, 9, 9)]


In [14]:
print(find_orfs("CCATGAAATGTT"))

[]


In [15]:
print(find_orfs("CCATGAAATAGTT"))

[(2, 11, 9)]


In [16]:
seq = data["NM_000207.3"]
orfs = find_orfs(seq)
for start, end, length in orfs:
    print("ORF from", start, "to", end, "| length", length)

ORF from 59 to 392 | length 333
ORF from 71 to 392 | length 321
ORF from 441 to 450 | length 9


## Step 7: Minimum length and the longest ORF
Short ORFs are usually random and are not real genes, so the program ignores any ORF shorter than a minimum length (default 60 letters). A second function returns the longest ORF, which in the insulin sequence is the full preproinsulin gene.

In [17]:
def find_orfs(dna, min_length=60):
    dna = dna.upper()
    stop_codons = ["TAA", "TAG", "TGA"]
    orfs = []

    for start in find_start_codons(dna):
        for i in range(start, len(dna) - 2, 3):
            codon = dna[i:i + 3]
            if codon in stop_codons:
                end = i + 3
                length = end - start
                if length >= min_length:
                    orfs.append((start, end, length))
                break

    return orfs

In [18]:
seq = data["NM_000207.3"]
orfs = find_orfs(seq)
for start, end, length in orfs:
    print("ORF from", start, "to", end, "| length", length)

ORF from 59 to 392 | length 333
ORF from 71 to 392 | length 321


In [19]:
print(find_orfs(seq, min_length=330))

[(59, 392, 333)]


In [20]:
def longest_orf(dna, min_length=60):
    orfs = find_orfs(dna, min_length)
    if len(orfs) == 0:
        return None
    return max(orfs, key=lambda orf: orf[2])

In [21]:
best = longest_orf(seq)
print(best)

(59, 392, 333)


In [22]:
start, end, length = best
protein = translate(seq[start:end])
print(protein)
print("Protein length:", len(protein) - 1, "amino acids")

MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN*
Protein length: 110 amino acids


## Step 8: Putting it together
A single function reads a FASTA file and, for each sequence, reports its length, the number of ORFs found, the longest ORF, and the protein it codes for. It handles missing files, empty sequences and sequences with no ORF.

In [23]:
def analyse_orfs(filename, min_length=60):
    try:
        data = read_fasta(filename)
    except FileNotFoundError:
        print("Oops, I could not find the file:", filename)
        return

    for name, sequence in data.items():
        print("Name:", name)
        print("Sequence length:", len(sequence))

        if len(sequence) == 0:
            print("This sequence is empty, skipping.")
            print()
            continue

        orfs = find_orfs(sequence, min_length)
        print("ORFs found (min length " + str(min_length) + "):", len(orfs))

        best = longest_orf(sequence, min_length)
        if best is None:
            print("No ORF long enough was found.")
        else:
            start, end, length = best
            protein = translate(sequence[start:end])
            print("Longest ORF: start", start, "| end", end, "| length", length)
            print("Protein length:", len(protein) - 1, "amino acids")
            print("Protein (first 30):", protein[:30])
        print()

In [24]:
analyse_orfs("real_sequence.fasta")

Name: NM_000207.3
Sequence length: 465
ORFs found (min length 60): 2
Longest ORF: start 59 | end 392 | length 333
Protein length: 110 amino acids
Protein (first 30): MALWMRLLPLLALLALWGPDPAAAFVNQHL



In [25]:
with open("test_orf.fasta", "w") as f:
    f.write(">tiny\nATGGCCTGA\n>none\nCCCCCC\n>empty\n")

analyse_orfs("test_orf.fasta", min_length=9)

Name: tiny
Sequence length: 9
ORFs found (min length 9): 1
Longest ORF: start 0 | end 9 | length 9
Protein length: 2 amino acids
Protein (first 30): MA*

Name: none
Sequence length: 6
ORFs found (min length 9): 0
No ORF long enough was found.

Name: empty
Sequence length: 0
This sequence is empty, skipping.



In [26]:
analyse_orfs("test_orf.fasta")

Name: tiny
Sequence length: 9
ORFs found (min length 60): 0
No ORF long enough was found.

Name: none
Sequence length: 6
ORFs found (min length 60): 0
No ORF long enough was found.

Name: empty
Sequence length: 0
This sequence is empty, skipping.



In [27]:
analyse_orfs("not_a_real_file.fasta")

Oops, I could not find the file: not_a_real_file.fasta


## Step 9: Validation
The translated protein from the longest ORF was compared with the human preproinsulin sequence from UniProt (P01308). The two sequences match exactly (110 amino acids), confirming that the program finds and translates the gene correctly.

In [28]:
seq = data["NM_000207.3"]
start, end, length = longest_orf(seq)
my_protein = translate(seq[start:end]).replace("*", "")

uniprot_protein = "MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN"

print("My protein length:", len(my_protein))
print("UniProt length:", len(uniprot_protein))
print("Match:", my_protein == uniprot_protein)

My protein length: 110
UniProt length: 110
Match: True


In [29]:
  for i in range(min(len(my_protein), len(uniprot_protein))):
      if my_protein[i] != uniprot_protein[i]:
          print("First difference at position", i)
          break

In [30]:
print("B chain:", my_protein[24:54])
print("A chain:", my_protein[89:110])

B chain: FVNQHLCGSHLVEALYLVCGERGFFYTPKT
A chain: GIVEQCCTSICSLYQLENYCN


## Step 10: Error handling
The program handles a missing file, a file with no sequences, empty sequences, sequences with no ORF, lowercase letters, and unexpected characters such as N. Unexpected characters trigger a warning, and codons that contain them appear as X in the protein.

This replaces the version from Step 8

In [31]:
def find_invalid_bases(sequence):
    valid = set("ATGC")
    return set(sequence.upper()) - valid

In [32]:
print(find_invalid_bases("ATGNCCTGA"))
print(find_invalid_bases("ATGC"))

{'N'}
set()


In [33]:
def analyse_orfs(filename, min_length=60):
    try:
        data = read_fasta(filename)
    except FileNotFoundError:
        print("Oops, I could not find the file:", filename)
        return

    # NEW: no sequences at all
    if len(data) == 0:
        print("No sequences found. Check that the file has header lines starting with '>'.")
        return

    for name, sequence in data.items():
        print("Name:", name)
        print("Sequence length:", len(sequence))

        if len(sequence) == 0:
            print("This sequence is empty, skipping.")
            print()
            continue

        # NEW: warn about unexpected characters
        invalid = find_invalid_bases(sequence)
        if invalid:
            print("Warning: unexpected characters found:", sorted(invalid))

        orfs = find_orfs(sequence, min_length)
        print("ORFs found (min length " + str(min_length) + "):", len(orfs))

        best = longest_orf(sequence, min_length)
        if best is None:
            print("No ORF long enough was found.")
        else:
            start, end, length = best
            protein = translate(sequence[start:end])
            print("Longest ORF: start", start, "| end", end, "| length", length)
            print("Protein length:", len(protein) - 1, "amino acids")
            print("Protein (first 30):", protein[:30])
        print()

In [34]:
analyse_orfs("real_sequence.fasta")

Name: NM_000207.3
Sequence length: 465
ORFs found (min length 60): 2
Longest ORF: start 59 | end 392 | length 333
Protein length: 110 amino acids
Protein (first 30): MALWMRLLPLLALLALWGPDPAAAFVNQHL



In [35]:
with open("test_n.fasta", "w") as f:
    f.write(">has_n\nATGNCCTGA\n")

analyse_orfs("test_n.fasta", min_length=9)

Name: has_n
Sequence length: 9
ORFs found (min length 9): 1
Longest ORF: start 0 | end 9 | length 9
Protein length: 2 amino acids
Protein (first 30): MX*



In [36]:
with open("test_lower.fasta", "w") as f:
    f.write(">lower\natggcctga\n")

analyse_orfs("test_lower.fasta", min_length=9)

Name: lower
Sequence length: 9
ORFs found (min length 9): 1
Longest ORF: start 0 | end 9 | length 9
Protein length: 2 amino acids
Protein (first 30): MA*



In [37]:
with open("test_noheader.fasta", "w") as f:
    f.write("ATGGCCTGA\n")

analyse_orfs("test_noheader.fasta")

No sequences found. Check that the file has header lines starting with '>'.


In [38]:
with open("test_blank.fasta", "w") as f:
    f.write("")

analyse_orfs("test_blank.fasta")

No sequences found. Check that the file has header lines starting with '>'.


## Example

Input: human insulin mRNA (NM_000207.3) from NCBI, 465 letters.

Output:
```
Name: NM_000207.3
Sequence length: 465
ORFs found (min length 60): 2
Longest ORF: start 59 | end 392 | length 333
Protein length: 110 amino acids
Protein (first 30): MALWMRLLPLLALLALWGPDPAAAFVNQHL
```



## Limitations
- Only searches the forward strand, so genes on the reverse strand are missed.
- Only uses ATG as a start codon. Some genes start with other codons.
- Does not handle introns, so it suits mRNA or cDNA better than raw genomic DNA.
- Uses the standard genetic code only (not the mitochondrial code).
- The default minimum length of 60 is arbitrary. Real analyses often use a longer cutoff.
- Nested ORFs that share a stop codon are reported separately.
- Codons containing unexpected characters such as N are translated as X.

## How this could be extended
- Search the reverse complement as well, giving six reading frames in total (three on each strand).
- Use Biopython (`Seq.translate()` and `SeqIO`) and compare its results with this program.
- Report only the longest ORF per stop codon, to avoid nested duplicates.
- Compare results with NCBI's ORFfinder tool.
- Handle genomic DNA with introns.
- Calculate GC content for each ORF and plot ORF lengths with matplotlib.